# Wind Speed Histogram Upload And Plot Demo

This notebook mirrors the structure of `scripts/1.0.lifetime-design-frequencies.ipynb` for the wind speed histogram workflow.

It parses the example workbook, builds validated histogram result series, optionally uploads backend-compatible rows, retrieves them back through `ResultsService`, and renders the package histogram plot.

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import enum
import math
import pathlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

if not hasattr(enum, "StrEnum"):
    class _CompatStrEnum(str, enum.Enum):
        pass
    enum.StrEnum = _CompatStrEnum

WORKSPACE_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(WORKSPACE_ROOT / "src"))
sys.path.insert(0, str(WORKSPACE_ROOT.parent / "owi-metadatabase-sdk" / "src"))

from owi.metadatabase.locations.io import LocationsAPI
from tqdm.notebook import tqdm as tqdm_notebook

from owi.metadatabase.results import (
    AnalysisDefinition,
    ResultsAPI,
    ResultsService,
    WindSpeedHistogram,
)
from owi.metadatabase.results.models import ResultSeries
from owi.metadatabase.results.serializers import DjangoAnalysisSerializer, DjangoResultSerializer
from owi.metadatabase.results.services import ApiResultsRepository

DEFAULT_WORKBOOK = WORKSPACE_ROOT / "scripts" / "data" / "results-example-data.xlsx"
print(f"Using workbook at {pathlib.Path(DEFAULT_WORKBOOK).resolve()}")
DEFAULT_BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
TOKEN_ENV_VAR = "OWI_METADATABASE_API_TOKEN"
POST_DATA = False


Using workbook at /home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/data/results-example-data.xlsx


In [ ]:
BASE_URL = DEFAULT_BASE_URL
WORKBOOK = DEFAULT_WORKBOOK
PROJECTSITE = "Belwind"
TOKEN = ""
LIVE_UPLOAD = TOKEN is not None
ANALYSIS_ID_OVERRIDE: int | None = None
analysis_id = ANALYSIS_ID_OVERRIDE

print({
    "workbook": str(WORKBOOK),
    "projectsite": PROJECTSITE,
    "live_upload": LIVE_UPLOAD,
    "analysis_id_override": ANALYSIS_ID_OVERRIDE,
})


{'workbook': '/home/pietro.dantuono@24SEA.local/Projects/SMARTLIFE/OWI-metadatabase-SDK/owi-metadatabase-results-sdk/scripts/data/results-example-data.xlsx', 'projectsite': 'Belwind', 'live_upload': True, 'analysis_id_override': None}


In [9]:
def resolve_site_and_locations(
    api_root: str, token: str | None, projectsite: str, labels: set[str], live: bool
) -> tuple[int | None, dict[str, int | None], pd.DataFrame]:
    if not live or token is None:
        return (
            1,
            {label: None for label in sorted(labels)},
            pd.DataFrame(columns=["id", "title", "northing", "easting"]),
        )
    locations_api = LocationsAPI(api_root=api_root, token=token)
    site_id = int(locations_api.get_projectsite_detail(projectsite=projectsite)["id"])
    assetlocations = locations_api.get_assetlocations(projectsite=projectsite)["data"]
    location_frame = assetlocations.loc[:, [
        column
        for column in ["id", "title", "northing", "easting"]
        if column in assetlocations.columns
    ]].copy()
    resolved_locations = {
        str(row["title"]): int(row["id"])
        for row in location_frame.to_dict(orient="records")
        if row.get("title") is not None and row.get("id") is not None
    }
    mapping = {label: resolved_locations.get(label) for label in sorted(labels)}
    return site_id, mapping, location_frame


class StubRepository:
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame

    def list_results(self, query):
        return self.frame


def normalize_bin_label(label: str) -> str:
    return str(label).replace("\n", "").replace(" ", "")


def parse_bin_edges(label: str) -> tuple[float, float]:
    normalized = normalize_bin_label(label).removeprefix("[").removesuffix("[")
    left, right = normalized.split(",", 1)
    return float(left), float(right)


def is_finite_payload(payload: dict) -> bool:
    for key in ["value_col1", "value_col2", "value_col3"]:
        values = payload.get(key)
        if values is None:
            continue
        if any(not math.isfinite(float(value)) for value in values):
            return False
    return True


wind_speed_histogram_analysis = WindSpeedHistogram()


def parse_histogram_sheet(
    path: Path, site_id: int, location_map: dict[str, int | None]
) -> tuple[AnalysisDefinition, list[ResultSeries]]:
    frame = pd.read_excel(path, sheet_name="Lifetime - Wind Histogram", header=1)
    bin_columns = [
        column
        for column in frame.columns
        if isinstance(column, str) and column.startswith("[")
    ]
    series = []
    for row in frame.to_dict(orient="records"):
        scope_label = str(row["Scope"]).strip()
        series.append(
            {
                "title": str(row["Title"]),
                "description": row.get("Description"),
                "scope_label": scope_label,
                "site_id": site_id,
                "location_id": location_map.get(scope_label),
                "bins": [parse_bin_edges(column) for column in bin_columns],
                "values": [float(row[column]) for column in bin_columns],
                "metadata": {"sheet_scope": scope_label},
            }
        )

    analysis = AnalysisDefinition(
        name=wind_speed_histogram_analysis.analysis_name,
        source_type="notebook",
        description="Workbook upload for the wind speed histogram sheet.",
        additional_data={"sheet_name": "Lifetime - Wind Histogram"},
    )
    return analysis, wind_speed_histogram_analysis.to_results({"series": series})


In [5]:
histogram_frame = pd.read_excel(WORKBOOK, sheet_name="Lifetime - Wind Histogram", header=1)
scope_labels = {str(value).strip() for value in histogram_frame["Scope"].dropna().tolist() if str(value).strip() != "Site"}
site_id, location_map, location_frame = resolve_site_and_locations(
    BASE_URL, TOKEN, PROJECTSITE, scope_labels, LIVE_UPLOAD
)
if not site_id:
    print("No site ID could be resolved, cannot proceed with upload.")
    site_id = -1
display(pd.DataFrame([
    {"scope": scope_label, "location_id": location_map[scope_label]}
    for scope_label in sorted(location_map)
]).head())


,scope,location_id
0,WTG1,None
1,WTG2,None


In [6]:
payload_sets = [
    parse_histogram_sheet(WORKBOOK, site_id, location_map),
]

summary_rows = [
    {
        "analysis": analysis_definition.name,
        "results": len(result_series),
        "sheet": analysis_definition.additional_data["sheet_name"],
    }
    for analysis_definition, result_series in payload_sets
]

display(pd.DataFrame(summary_rows))


,analysis,results,sheet
0,WindSpeedHistogram,5,Lifetime - Wind Histogram


In [7]:
analysis_serializer = DjangoAnalysisSerializer()
result_serializer = DjangoResultSerializer()

preview_rows = []
for analysis_definition, result_series in payload_sets:
    result_payloads = [result_serializer.to_payload(item, analysis_id=0) for item in result_series]
    invalid_payloads = [payload for payload in result_payloads if payload.get("location") is None]
    valid_payloads = [payload for payload in result_payloads if payload.get("location") is not None]
    preview_rows.append(
        {
            "analysis": analysis_definition.name,
            "uploadable_rows": len(valid_payloads),
            "skipped_rows": len(invalid_payloads),
            "first_result": result_series[0].short_description,
            "site_scoped_rows": sum(payload.get("location") is None for payload in result_payloads),
        }
    )

display(pd.DataFrame(preview_rows))


,analysis,uploadable_rows,skipped_rows,first_result,site_scoped_rows
0,WindSpeedHistogram,0,5,Design,5


## Data Upload

The next cell uploads only the rows that can be resolved to a backend `location` and have finite numeric values. Site-scoped rows remain useful for local validation but are skipped by the current dev backend flow.

In [8]:
api = ResultsAPI(api_root=BASE_URL, token=TOKEN)
upload_log = []
if not LIVE_UPLOAD or not POST_DATA:
    print("Upload skipped. Set POST_DATA = True to create backend rows.")
else:
    for analysis_def, result_series in payload_sets:
        result_payloads = [result_serializer.to_payload(item, analysis_id=0) for item in result_series]
        valid_payloads = [
            payload
            for payload in result_payloads
            if payload.get("location") is not None and is_finite_payload(payload)
        ]
        if not valid_payloads:
            upload_log.append({"analysis": analysis_def.name, "uploaded_rows": 0, "status": "skipped"})
            continue

        created_analysis = api.create_analysis(analysis_serializer.to_payload(analysis_def))
        analysis_id = int(created_analysis["id"])
        print(f"Created analysis with ID {analysis_id} for {analysis_def.name}.")
        upload_payloads = [{**payload, "analysis": analysis_id} for payload in valid_payloads]
        for payload in tqdm_notebook(upload_payloads, desc=f"Uploading results for {analysis_def.name}"):
            response = api.create_result(payload)
        upload_log.append(
            {
                "analysis": analysis_def.name,
                "uploaded_rows": len(upload_payloads),
                "status": bool(response["exists"]),
            }
        )

display(pd.DataFrame(upload_log))


Upload skipped. Set POST_DATA = True to create backend rows.


""


## Retrieve And Plot Uploaded Wind Speed Histograms

The next cells fetch the uploaded `WindSpeedHistogram` rows back from the backend, reconstruct the normalized histogram table with the results package, and render the high-level histogram plot through `ResultsService`.

In [10]:
if analysis_id is None:
    local_frame = pd.DataFrame([
        item.to_record_payload(analysis_id=999)
        for item in payload_sets[0][1]
    ])
    results_service = ResultsService(repository=StubRepository(local_frame))
    retrieved_series = results_service.get_result_series(
        wind_speed_histogram_analysis.analysis_name,
    )
    retrieved_histograms = wind_speed_histogram_analysis.from_results(retrieved_series)
    print("Using in-memory payloads because no backend analysis_id was configured.")
else:
    retrieval_api = api if "api" in globals() else ResultsAPI(api_root=BASE_URL, token=TOKEN)
    results_service = ResultsService(repository=ApiResultsRepository(api=retrieval_api))
    retrieved_series = results_service.get_result_series(
        wind_speed_histogram_analysis.analysis_name,
        filters={"analysis_id": analysis_id},
    )
    retrieved_histograms = wind_speed_histogram_analysis.from_results(retrieved_series)
retrieved_histograms.head()


Using in-memory payloads because no backend analysis_id was configured.


,series_name,scope,bin_left,bin_right,value
0,Design,Site,0.0,1.0,1.0
1,Design,Site,1.0,2.0,2.0
2,Design,Site,2.0,3.0,3.0
3,Design,Site,3.0,4.0,4.0
4,Design,Site,4.0,5.0,5.0


In [11]:
location_ids = sorted({series.location_id for series in retrieved_series if series.location_id is not None})
location_lookup = results_service.get_location_frame(location_ids) if retrieved_series else pd.DataFrame()
storage_diagnostics = pd.DataFrame(
    [
        {
            "short_description": series.short_description,
            "scope": series.result_scope.value,
            "scope_label": series.data_additional.get("scope_label"),
            "vector_names": [vector.name for vector in series.vectors],
            "bin_count": len(series.vectors[0].values),
            "location_id": series.location_id,
        }
        for series in retrieved_series
    ]
)

display(storage_diagnostics)
if not location_lookup.empty:
    display(location_lookup.head())


,short_description,scope,scope_label,vector_names,bin_count,location_id
0,Design,site,Site,"[bin_left, value, bin_right]",50,None
1,WTG1 - Y1,site,WTG1,"[bin_left, value, bin_right]",50,None
2,WTG2 - Y1,site,WTG2,"[bin_left, value, bin_right]",50,None
3,WTG1 - Y2,site,WTG1,"[bin_left, value, bin_right]",50,None
4,WTG2 - Y2,site,WTG2,"[bin_left, value, bin_right]",50,None


In [14]:
histogram_plot = results_service.plot_results(
    wind_speed_histogram_analysis.analysis_name,
    filters={"analysis_id": analysis_id} if analysis_id is not None else None,
    plot_type="histogram",
)
display(histogram_plot.notebook)


In [15]:
plot_summary = pd.DataFrame(
    [
        {
            "retrieved_series": len(retrieved_series),
            "retrieved_rows": len(retrieved_histograms),
            "series_names": sorted(retrieved_histograms["series_name"].unique().tolist()) if not retrieved_histograms.empty else [],
            "histogram_plot_available": histogram_plot is not None and histogram_plot.notebook is not None,
        }
    ]
)
display(plot_summary)


,retrieved_series,retrieved_rows,series_names,histogram_plot_available
0,5,250,"[Design, WTG1 - Y1, WTG1 - Y2, WTG2 - Y1, WTG2...",True
